In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import roman_datamodels as rdm

from astropy.visualization import simple_norm
from photutils.detection import IRAFStarFinder
from photutils.background import MMMBackground, MADStdBackgroundRMS
from photutils.aperture import CircularAperture
from ipywidgets import interact_manual, FloatSlider

In [5]:
# 1. Load Data
file = rdm.open('car086_wfi08_f146_perturbed_cal.asdf')
data_es = file.data * 2 # Gain = 2

# 2. Calculate Background once (this is computationally heavy)
bkgrms = MADStdBackgroundRMS()
mmm_bkg = MMMBackground()
std = bkgrms(data_es)
bkg = mmm_bkg(data_es)

norm = simple_norm(data_es, 'asinh', vmin=0.1, vmax=20)
print(f"Background: {bkg:.2f}, StdDev: {std:.2f}")

Background: 2.49, StdDev: 0.40


In [6]:
@interact_manual(
    sigma=FloatSlider(value=50.0, min=5.0, max=100.0, step=5.0, description='Bkg Sigma:'),
    fwhm=FloatSlider(value=1.5, min=1.0, max=10.0, step=0.5, description='FWHM:'),
    sharp_lo=FloatSlider(value=0.6, min=-1.0, max=1.0, step=0.1, description='Sharp Lo:'),
    sharp_hi=FloatSlider(value=1.4, min=1.0, max=3.0, step=0.1, description='Sharp Hi:'),
    round_hi=FloatSlider(value=0.6, min=0.1, max=2.0, step=0.1, description='Round Hi:'),
    min_flux=FloatSlider(value=50.0, min=10.0, max=500.0, step=10.0, description='Min Flux:')
)
def tune_photometry(sigma, fwhm, sharp_lo, sharp_hi, round_hi, min_flux):
    
    # 1. Run Finder with updated Photutils 3.0 syntax
    finder = IRAFStarFinder(
        threshold=sigma * std + bkg, 
        fwhm=fwhm, 
        min_separation=7.0 * fwhm,                  # Replaces minsep_fwhm
        roundness_range=(-round_hi, round_hi),      # Replaces roundlo/roundhi
        sharpness_range=(sharp_lo, sharp_hi)        # Replaces sharplo/sharphi
    )
    sources = finder(data_es)
    
    if sources is None:
        print("No sources found with these parameters!")
        return
        
    # 2. Apply Flux Mask
    mask = sources['flux'] > min_flux
    sources_masked = sources[mask]
    print(f"Sources Found: {len(sources)} | Sources after flux cut: {len(sources_masked)}")
    
    # 3. Plot Diagnostics (Redesigned for a massive image view)
    fig = plt.figure(figsize=(18, 20))
    gs = gridspec.GridSpec(4, 3, height_ratios=[1, 3, 3, 3]) 
    
    # Top Row: Scatters
    ax_sharp = fig.add_subplot(gs[0, 0])
    ax_round = fig.add_subplot(gs[0, 1])
    ax_flux = fig.add_subplot(gs[0, 2])
    
    # Bottom Rows: Massive Image Plot
    ax_img = fig.add_subplot(gs[1:, :])

    # Scatters
    ax_sharp.plot(sources['mag'], sources['sharpness'], 'ko', markersize=2, alpha=0.3)
    ax_sharp.plot(sources_masked['mag'], sources_masked['sharpness'], 'ro', markersize=3)
    ax_sharp.set_xlabel('Mag'); ax_sharp.set_ylabel('Sharpness')
    
    ax_round.plot(sources['mag'], sources['roundness'], 'ko', markersize=2, alpha=0.3)
    ax_round.plot(sources_masked['mag'], sources_masked['roundness'], 'ro', markersize=3)
    ax_round.set_xlabel('Mag'); ax_round.set_ylabel('Roundness')
    
    ax_flux.plot(sources['flux'], sources['fwhm'], 'ko', markersize=2, alpha=0.3)
    ax_flux.plot(sources_masked['flux'], sources_masked['fwhm'], 'ro', markersize=3)
    ax_flux.set_xlabel('Flux'); ax_flux.set_ylabel('FWHM')
    ax_flux.set_xlim(0, max(2000, np.percentile(sources['flux'], 95)))
    
    # Image Cutout
    ax_img.imshow(data_es, norm=norm, cmap='Greys', origin='lower')
    
    # Updated to x_centroid and y_centroid
    positions = np.transpose((sources_masked['x_centroid'], sources_masked['y_centroid']))
    apertures = CircularAperture(positions, r=10)
    apertures.plot(color='blue', lw=2.0, alpha=0.8, ax=ax_img)
    
    # A tighter zoom to make the sources highly visible
    ax_img.set_xlim(1500, 2500)
    ax_img.set_ylim(1500, 2500)
    ax_img.set_title("Central 1000x1000 px Crop with Validated Apertures")
    
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=50.0, description='Bkg Sigma:', min=5.0, step=5.0), FloatSlider(value=…

In [2]:
# Grab the current values directly from the interactive widget's state
# (Make sure to set these to whatever your final slider values are)
optimal_params = {
    "sigma_threshold": 50.0,
    "fwhm": 3.5, 
    "sharp_lo": 0.6,
    "sharp_hi": 1.4,
    "round_hi": 0.6,
    "min_flux": 50.0
}

with open('car086_phot_config.json', 'w') as f:
    json.dump(optimal_params, f, indent=4)
    
print("Configuration saved to car086_phot_config.json!")

Configuration saved to car086_phot_config.json!
